<a href="https://colab.research.google.com/github/isiescalona/CSN/blob/main/pandas_ciencias_dedatos_II.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ANÁLISIS CON PANDAS Y KAGGLE
# Código completo para Google Colab, sin gráficos.
#
# Dataset elegido: Retail Sales Dataset.
# Descarga: https://www.kaggle.com/datasets/mohammadtalib786/retail-sales-dataset
#
# Descarga y descomprime el dataset.
# Ejecuta este código y selecciona el archivo CSV.
# No es necesario cambiar su nombre a dataset.csv.
#
# Google Colab ya incluye Pandas.

import io
import pandas as pd
from google.colab import files
from IPython.display import display


# --------------------------------------------------
# 1. CARGAR LOS DATOS
# --------------------------------------------------

print("Selecciona el archivo CSV del dataset de ventas.")
archivos = files.upload()

if len(archivos) != 1:
    raise ValueError("Debes seleccionar un solo archivo.")

nombre_archivo = list(archivos.keys())[0]

if not nombre_archivo.lower().endswith(".csv"):
    raise ValueError("El archivo debe tener extensión .csv.")

datos = pd.read_csv(io.BytesIO(archivos[nombre_archivo]))

# Quitar espacios de los nombres de las columnas.
datos.columns = datos.columns.str.strip()

columnas_necesarias = [
    "Transaction ID",
    "Date",
    "Customer ID",
    "Gender",
    "Age",
    "Product Category",
    "Quantity",
    "Price per Unit",
    "Total Amount"
]

faltantes = [
    columna
    for columna in columnas_necesarias
    if columna not in datos.columns
]

if faltantes:
    raise ValueError(
        "El archivo no corresponde al dataset esperado. "
        f"Faltan estas columnas: {', '.join(faltantes)}"
    )

# Usar nombres en español para facilitar la lectura.
datos = datos.rename(columns={
    "Transaction ID": "Id_Transaccion",
    "Date": "Fecha",
    "Customer ID": "Id_Cliente",
    "Gender": "Genero",
    "Age": "Edad",
    "Product Category": "Categoria",
    "Quantity": "Cantidad",
    "Price per Unit": "Precio",
    "Total Amount": "Total_Original"
})

print("\nPRIMERAS 10 FILAS")
display(datos.head(10))


# --------------------------------------------------
# 2. EXPLORACIÓN INICIAL
# --------------------------------------------------

print("\nÚLTIMAS 5 FILAS")
display(datos.tail(5))

print("\nINFORMACIÓN GENERAL")
datos.info()

print("\nESTADÍSTICAS DESCRIPTIVAS")
display(datos.describe())

print("\nTIPOS DE DATOS")
print(datos.dtypes)

print("\nVALORES FALTANTES ORIGINALES")
print(datos.isna().sum())

filas_originales = len(datos)


# --------------------------------------------------
# 3. LIMPIEZA DE DATOS
# --------------------------------------------------

# Limpiar espacios de las columnas de texto.
# Los textos vacíos se consideran datos faltantes.
columnas_texto = [
    "Id_Transaccion",
    "Id_Cliente",
    "Genero",
    "Categoria"
]

for columna in columnas_texto:
    datos[columna] = datos[columna].astype("string").str.strip()
    datos[columna] = datos[columna].replace("", pd.NA)

# Convertir las fechas.
# Los valores que no se puedan interpretar quedarán como NaT.
datos["Fecha"] = pd.to_datetime(
    datos["Fecha"],
    format="%Y-%m-%d",
    errors="coerce"
)

# Convertir las columnas numéricas.
# Los textos inválidos quedarán como NaN.
columnas_numericas = [
    "Edad",
    "Cantidad",
    "Precio",
    "Total_Original"
]

for columna in columnas_numericas:
    datos[columna] = pd.to_numeric(
        datos[columna],
        errors="coerce"
    )

    # Los valores infinitos tampoco sirven para este análisis.
    datos[columna] = datos[columna].replace(
        [float("inf"), float("-inf")],
        float("nan")
    )

print("\nVALORES FALTANTES DESPUÉS DE CONVERTIR LOS TIPOS")
print(datos.isna().sum())

# Eliminar únicamente filas completamente duplicadas.
duplicados = datos.duplicated().sum()
datos = datos.drop_duplicates().copy()

print("\nFilas duplicadas eliminadas:", duplicados)

# Un identificador repetido con datos distintos requiere revisión.
# No elegimos una de esas transacciones de forma arbitraria.
conflictos = datos["Id_Transaccion"].duplicated(keep=False)
conflictos = conflictos & datos["Id_Transaccion"].notna()

if conflictos.any():
    display(datos.loc[conflictos])
    raise ValueError(
        "Hay identificadores de transacción repetidos con datos distintos. "
        "Revisa esas filas antes de continuar."
    )

# Para analizar ventas necesitamos fecha, categoría, cantidad y precio.
# No rellenamos cantidades o precios con promedios:
# eso podría inventar ingresos que no ocurrieron.
columnas_esenciales = [
    "Id_Transaccion",
    "Fecha",
    "Categoria",
    "Cantidad",
    "Precio"
]

filas_antes = len(datos)
datos = datos.dropna(subset=columnas_esenciales).copy()

print(
    "Filas eliminadas por datos esenciales faltantes:",
    filas_antes - len(datos)
)

# En este ejercicio analizamos ventas, no devoluciones.
# La cantidad debe ser un entero positivo y el precio no puede ser negativo.
ventas_validas = (
    (datos["Cantidad"] > 0)
    & (datos["Cantidad"] % 1 == 0)
    & (datos["Precio"] >= 0)
)

print("Filas con cantidad o precio inválidos:", (~ventas_validas).sum())

datos = datos.loc[ventas_validas].copy()

if datos.empty:
    raise ValueError("No quedaron ventas válidas para analizar.")

datos["Cantidad"] = datos["Cantidad"].astype(int)

# No inventamos identificadores de clientes.
# Para género, usamos una etiqueta que indica falta de información.
datos["Genero"] = datos["Genero"].fillna("Sin información")

# Tratar edades negativas, cero o fraccionarias como faltantes.
edades_invalidas = (
    (datos["Edad"] <= 0)
    | (datos["Edad"] % 1 != 0)
)

datos.loc[edades_invalidas, "Edad"] = float("nan")

# Conservar una marca antes de rellenar las edades.
datos["Edad_Imputada"] = datos["Edad"].isna()

# Usar la mediana porque suele verse menos afectada por valores extremos.
mediana_edad = datos["Edad"].median()

if pd.notna(mediana_edad):
    datos["Edad"] = datos["Edad"].fillna(mediana_edad)
    print("Edades rellenadas con la mediana:", datos["Edad_Imputada"].sum())
    print("Mediana utilizada:", mediana_edad)
else:
    print("No hay edades válidas: se conservan como valores faltantes.")

# Ordenar por fecha y crear un índice consecutivo.
datos = datos.sort_values("Fecha").reset_index(drop=True)

print("\nRESUMEN DE LIMPIEZA")
print("Filas originales:", filas_originales)
print("Filas disponibles:", len(datos))
print("Identificadores de cliente faltantes:", datos["Id_Cliente"].isna().sum())

print("\nVALORES FALTANTES FINALES")
print(datos.isna().sum())


# --------------------------------------------------
# 4. TRANSFORMACIÓN DE DATOS
# --------------------------------------------------

# Calcular ingresos a partir de cantidad × precio.
datos["Ingresos"] = datos["Cantidad"] * datos["Precio"]

# Comparar con el importe original para detectar diferencias.
# Se permite una diferencia mínima por redondeo.
diferencias = (
    datos["Total_Original"].notna()
    & (
        (datos["Total_Original"] - datos["Ingresos"]).abs()
        > 0.01
    )
)

print("\nTRANSACCIONES CON DIFERENCIAS EN EL TOTAL:", diferencias.sum())

if diferencias.any():
    display(
        datos.loc[
            diferencias,
            ["Id_Transaccion", "Cantidad", "Precio", "Total_Original", "Ingresos"]
        ]
    )

print("Para los cálculos se utilizará Cantidad × Precio.")

# Crear columnas de tiempo para futuras agrupaciones.
datos["Anio"] = datos["Fecha"].dt.year
datos["Mes"] = datos["Fecha"].dt.month
datos["Periodo"] = datos["Fecha"].dt.to_period("M").astype(str)

# Normalización min-max: llevar los ingresos a una escala de 0 a 1.
# Esto cambia la escala, no los ingresos originales.
ingreso_minimo = datos["Ingresos"].min()
ingreso_maximo = datos["Ingresos"].max()

if ingreso_maximo > ingreso_minimo:
    datos["Ingresos_Normalizados"] = (
        (datos["Ingresos"] - ingreso_minimo)
        / (ingreso_maximo - ingreso_minimo)
    )
else:
    # Evitar dividir por cero si todos los ingresos son iguales.
    datos["Ingresos_Normalizados"] = 0.0

# Clasificar transacciones con límites de ejemplo.
# Estos límites son una decisión del ejercicio, no una regla del negocio.
datos["Nivel_Venta"] = pd.cut(
    datos["Ingresos"],
    bins=[-float("inf"), 100, 500, float("inf")],
    labels=["Baja", "Media", "Alta"]
)


# Función personalizada para usar con apply().
# Considera tanto el ingreso como la cantidad de unidades.
def clasificar_compra(fila):
    if fila["Ingresos"] > 500 and fila["Cantidad"] >= 3:
        return "Importe alto y varias unidades"
    elif fila["Ingresos"] > 500:
        return "Importe alto y pocas unidades"
    else:
        return "Importe hasta 500"


datos["Tipo_Compra"] = datos.apply(clasificar_compra, axis=1)

print("\nDATOS TRANSFORMADOS")
display(datos.head(10))


# --------------------------------------------------
# 5. AGRUPACIÓN Y AGREGACIÓN DE DATOS
# --------------------------------------------------

# Una fila representa una transacción.
# count cuenta ingresos no nulos; en los datos limpios equivale
# a la cantidad de transacciones.
resumen_categorias = datos.groupby("Categoria").agg(
    Ingresos_Totales=("Ingresos", "sum"),
    Ingreso_Promedio=("Ingresos", "mean"),
    Transacciones=("Ingresos", "count"),
    Ingreso_Minimo=("Ingresos", "min"),
    Ingreso_Maximo=("Ingresos", "max"),
    Desviacion_Estandar=("Ingresos", "std"),
    Varianza=("Ingresos", "var"),
    Unidades_Vendidas=("Cantidad", "sum")
)

resumen_categorias = resumen_categorias.sort_values(
    "Ingresos_Totales",
    ascending=False
)

print("\nRESUMEN POR CATEGORÍA")
display(resumen_categorias.round(2))

# std y var usan el cálculo muestral de Pandas.
# Si un grupo tiene una sola transacción, ambas serán NaN.

resumen_mensual = datos.groupby("Periodo").agg(
    Ingresos_Totales=("Ingresos", "sum"),
    Transacciones=("Ingresos", "count"),
    Unidades_Vendidas=("Cantidad", "sum")
)

print("\nRESUMEN POR MES")
display(resumen_mensual.round(2))

print("Los meses inicial y final podrían tener datos de solo parte del mes.")

resumen_niveles = datos.groupby(
    "Nivel_Venta",
    observed=True
).agg(
    Transacciones=("Ingresos", "count"),
    Ingresos_Totales=("Ingresos", "sum")
)

print("\nRESUMEN POR NIVEL DE VENTA")
display(resumen_niveles.round(2))

print("\nCANTIDAD DE TRANSACCIONES POR TIPO DE COMPRA")
print(datos["Tipo_Compra"].value_counts())


# --------------------------------------------------
# 6. EXTRAER CONCLUSIONES A PARTIR DE LOS RESULTADOS
# --------------------------------------------------

total_ingresos = datos["Ingresos"].sum()
total_unidades = datos["Cantidad"].sum()
promedio_transaccion = datos["Ingresos"].mean()

totales_categoria = resumen_categorias["Ingresos_Totales"]

categorias_mayores = totales_categoria[
    totales_categoria == totales_categoria.max()
].index

categorias_menores = totales_categoria[
    totales_categoria == totales_categoria.min()
].index

print("\nCONCLUSIONES DEL ANÁLISIS")
print("Período analizado:", datos["Fecha"].min().date(),
      "hasta", datos["Fecha"].max().date())
print(f"Ingresos totales: {total_ingresos:,.2f}")
print(f"Unidades vendidas: {total_unidades:,}")
print(f"Ingreso promedio por transacción: {promedio_transaccion:,.2f}")
print("Categoría con mayores ingresos:", ", ".join(categorias_mayores))
print("Categoría con menores ingresos:", ", ".join(categorias_menores))

if total_ingresos > 0:
    participacion = totales_categoria.max() / total_ingresos * 100
    print(
        "Participación de cada categoría con mayores ingresos: "
        f"{participacion:.2f}% del total."
    )

print("\nLos importes se muestran sin asignar una moneda.")
print("Los ingresos no representan ganancias: no tenemos los costos.")
print("Las conclusiones corresponden a las filas conservadas tras la limpieza.")


# --------------------------------------------------
# 7. GUARDAR LOS RESULTADOS
# --------------------------------------------------

datos.to_csv(
    "ventas_limpias.csv",
    index=False,
    encoding="utf-8-sig"
)

resumen_categorias.to_csv(
    "resumen_por_categoria.csv",
    encoding="utf-8-sig"
)

resumen_mensual.to_csv(
    "resumen_por_mes.csv",
    encoding="utf-8-sig"
)

print("\nARCHIVOS GENERADOS")
print("- ventas_limpias.csv")
print("- resumen_por_categoria.csv")
print("- resumen_por_mes.csv")
print("Puedes descargarlos desde el panel de archivos de Google Colab.")

Selecciona el archivo CSV del dataset de ventas.


Saving retail_sales_dataset.csv to retail_sales_dataset.csv

PRIMERAS 10 FILAS


,Id_Transaccion,Fecha,Id_Cliente,Genero,Edad,Categoria,Cantidad,Precio,Total_Original
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100
5,6,2023-04-25,CUST006,Female,45,Beauty,1,30,30
6,7,2023-03-13,CUST007,Male,46,Clothing,2,25,50
7,8,2023-02-22,CUST008,Male,30,Electronics,4,25,100
8,9,2023-12-13,CUST009,Male,63,Electronics,2,300,600
9,10,2023-10-07,CUST010,Female,52,Clothing,4,50,200



ÚLTIMAS 5 FILAS


,Id_Transaccion,Fecha,Id_Cliente,Genero,Edad,Categoria,Cantidad,Precio,Total_Original
995,996,2023-05-16,CUST996,Male,62,Clothing,1,50,50
996,997,2023-11-17,CUST997,Male,52,Beauty,3,30,90
997,998,2023-10-29,CUST998,Female,23,Beauty,4,25,100
998,999,2023-12-05,CUST999,Female,36,Electronics,3,50,150
999,1000,2023-04-12,CUST1000,Male,47,Electronics,4,30,120


...


RESUMEN POR MES


,Ingresos_Totales,Transacciones,Unidades_Vendidas
Periodo,,,
2023-01,35450,76,195
2023-02,44060,85,214
2023-03,28990,73,194
2023-04,33870,86,214
2023-05,53150,105,259
2023-06,36715,77,197
2023-07,35465,72,176
2023-08,36960,94,227
2023-09,23620,65,170


Los meses inicial y final podrían tener datos de solo parte del mes.

RESUMEN POR NIVEL DE VENTA


,Transacciones,Ingresos_Totales
Nivel_Venta,,
Baja,457,29240
Media,244,63160
Alta,299,363600



CANTIDAD DE TRANSACCIONES POR TIPO DE COMPRA
Tipo_Compra
Importe hasta 500                 701
Importe alto y varias unidades    215
Importe alto y pocas unidades      84
Name: count, dtype: int64

CONCLUSIONES DEL ANÁLISIS
Período analizado: 2023-01-01 hasta 2024-01-01
Ingresos totales: 456,000.00
Unidades vendidas: 2,514
Ingreso promedio por transacción: 456.00
Categoría con mayores ingresos: Electronics
Categoría con menores ingresos: Beauty
Participación de cada categoría con mayores ingresos: 34.41% del total.

Los importes se muestran sin asignar una moneda.
Los ingresos no representan ganancias: no tenemos los costos.
Las conclusiones corresponden a las filas conservadas tras la limpieza.

ARCHIVOS GENERADOS
- ventas_limpias.csv
- resumen_por_categoria.csv
- resumen_por_mes.csv
Puedes descargarlos desde el panel de archivos de Google Colab.
